# 实验 1: 超声仿真 — 点散射体模型与波束合成

本 notebook 演示完整的超声成像 pipeline：
1. 创建散射体 phantom（点目标、囊肿）
2. 模拟阵元 RF 数据采集
3. DAS 波束合成
4. B-mode 图像重建与可视化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from us_imaging.simulation.phantom import Phantom
from us_imaging.simulation.acquire import TransducerArray, simulate_rf
from us_imaging.beamforming.das import das_beamform
from us_imaging.reconstruction.envelope import envelope_hilbert
from us_imaging.reconstruction.compress import log_compress
from us_imaging.visualize.bmode import render_bmode

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Modules loaded.')

In [ ]:
# 配置64阵元换能器阵列
array = TransducerArray(
    n_elements=64,
    pitch=0.3e-3,         # 0.3 mm 阵元间距
    center_freq=5e6,      # 5 MHz 中心频率
    bandwidth=0.7,        # 70% 带宽
    sampling_freq=40e6,   # 40 MHz 采样率
)

wavelength = array.sound_speed / array.center_freq
print(f'中心频率: {array.center_freq/1e6:.1f} MHz')
print(f'波长: {wavelength*1e3:.3f} mm')
print(f'阵元间距: {array.pitch*1e3:.3f} mm ({array.pitch/wavelength:.2f}λ)')
print(f'阵列总宽度: {array.n_elements * array.pitch * 1e3:.1f} mm')

## 1.1 单点目标 — 验证波束合成正确性

In [ ]:
# 在 z=20mm 深度放置一个点散射体
phantom_pt = Phantom.point_target(x_m=0.0, z_m=0.020, amplitude=1.0)

# 模拟 RF 数据
rf_pt = simulate_rf(phantom_pt, array, t_start=0, t_end=50e-6)
print(f'RF data shape: {rf_pt.shape} (Tx x Rx x Samples)')

# 查看单通道 RF 信号
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
tx_mid = 32
ax1.imshow(rf_pt[tx_mid], aspect='auto', cmap='RdBu_r', vmin=-0.1, vmax=0.1)
ax1.set_title(f'RF: Tx={tx_mid}, All Rx channels')
ax1.set_xlabel('Time sample')
ax1.set_ylabel('Rx element')

ax2.plot(rf_pt[tx_mid, tx_mid, :500])
ax2.set_title(f'RF trace: Tx={tx_mid}, Rx={tx_mid} (pulse-echo)')
ax2.set_xlabel('Sample')
plt.tight_layout()
plt.show()

In [ ]:
# 定义成像网格
x_grid = np.linspace(-0.01, 0.01, 256)   # ±10mm
z_grid = np.linspace(0.005, 0.04, 512)    # 5~40mm depth

# DAS 波束合成
bf_pt = das_beamform(rf_pt, array, x_grid, z_grid, tx_idx=32, f_number=1.5)

# 包络 + 对数压缩
env_pt = envelope_hilbert(bf_pt)
bmode_pt = log_compress(env_pt, dynamic_range=60)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
render_bmode(bf_pt, [x_grid[0], x_grid[-1], z_grid[-1], z_grid[0]],
            title='Raw Beamformed', ax=axes[0])
render_bmode(env_pt, [x_grid[0], x_grid[-1], z_grid[-1], z_grid[0]],
            title='Envelope (Hilbert)', ax=axes[1])
render_bmode(bmode_pt, [x_grid[0], x_grid[-1], z_grid[-1], z_grid[0]],
            title='B-mode (Log Compressed 60dB)', ax=axes[2])
plt.tight_layout()
plt.show()

# 横向分辨率: z=20mm 处的横向剖面
z_idx = np.argmin(np.abs(z_grid - 0.020))
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(x_grid * 1e3, bmode_pt[z_idx])
ax.set_xlabel('Lateral (mm)')
ax.set_ylabel('Normalized amplitude')
ax.set_title(f'Lateral profile at z=20mm')
ax.axvline(0, color='r', ls='--', alpha=0.5)
plt.show()

## 1.2 点阵 Phantom — 评估空间分辨率

In [ ]:
# 3x3 点阵，间距 5mm (横向) x 10mm (纵向)
phantom_grid = Phantom.point_grid(n_x=3, n_z=3, dx=0.005, dz=0.01, z_start=0.01)

rf_grid = simulate_rf(phantom_grid, array, t_start=0, t_end=55e-6)

# 完整 pipeline
bf_grid = das_beamform(rf_grid, array, x_grid, z_grid, tx_idx=32)
env_grid = envelope_hilbert(bf_grid)
bmode_grid = log_compress(env_grid, dynamic_range=60)

render_bmode(bmode_grid, [x_grid[0], x_grid[-1], z_grid[-1], z_grid[0]],
            title='3x3 Point Grid — B-mode (60 dB)')

## 1.3 囊肿 Phantom — 模拟软组织成像

In [ ]:
# 创建模拟囊肿
phantom_cyst = Phantom.cyst(
    center_x=0.0, center_z=0.025, radius=0.003,
    n_scatterers=500,
    interior_amp=0.1,   # 囊肿内部 = 低回声
    exterior_amp=0.6,   # 周围组织 = 高回声
    region_width=0.02, region_depth=0.02
)
print(f'Cyst phantom: {len(phantom_cyst)} scatterers')

# 可视化散射体分布
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(phantom_cyst.x * 1e3, phantom_cyst.z * 1e3,
               c=phantom_cyst.amplitude, s=10, cmap='gray_r', edgecolors='none')
ax.set_xlabel('Lateral (mm)')
ax.set_ylabel('Depth (mm)')
ax.set_title('Cyst Phantom — Scatterer Distribution')
ax.invert_yaxis()
plt.colorbar(sc, label='Amplitude')
plt.show()

In [ ]:
# 模拟 RF 并重建
rf_cyst = simulate_rf(phantom_cyst, array, t_start=0, t_end=55e-6)
print(f'RF shape: {rf_cyst.shape}')

z_grid = np.linspace(0.005, 0.045, 512)
bf_cyst = das_beamform(rf_cyst, array, x_grid, z_grid, tx_idx=32)
env_cyst = envelope_hilbert(bf_cyst)
bmode_cyst = log_compress(env_cyst, dynamic_range=60)

render_bmode(bmode_cyst, [x_grid[0], x_grid[-1], z_grid[-1], z_grid[0]],
            title='Cyst Phantom — B-mode (60 dB)')

## 1.4 平面波成像 (Plane Wave Imaging)

In [ ]:
from us_imaging.simulation.acquire import plane_wave_rf
from us_imaging.beamforming.das import das_beamform_plane_wave

# 3 角度平面波 (0°, ±5°)
angles = np.radians([0, -5, 5])

phantom_pw = Phantom.point_grid(n_x=5, n_z=3, dx=0.003, dz=0.008, z_start=0.01)
rf_pw = plane_wave_rf(phantom_pw, array, angles=angles, t_start=0, t_end=55e-6)
print(f'Plane wave RF: {rf_pw.shape} (angles x elements x samples)')

# 对每个角度分别做 DAS，然后相干复合
bf_compound = np.zeros((len(z_grid), len(x_grid)), dtype=np.float32)
for ai in range(len(angles)):
    bf_angle = das_beamform_plane_wave(
        rf_pw[ai], array, x_grid, z_grid,
        angle=angles[ai], f_number=1.5
    )
    bf_compound += bf_angle

env_pw = envelope_hilbert(bf_compound)
bmode_pw = log_compress(env_pw, dynamic_range=60)

fig, axes = plt.subplots(1, len(angles) + 1, figsize=(20, 5))
for ai in range(len(angles)):
    bmode_i = log_compress(envelope_hilbert(das_beamform_plane_wave(
        rf_pw[ai], array, x_grid, z_grid, angle=angles[ai])), dynamic_range=60)
    render_bmode(bmode_i, [x_grid[0], x_grid[-1], z_grid[-1], z_grid[0]],
                title=f'PWI Angle {np.degrees(angles[ai]):.0f}°', ax=axes[ai])
render_bmode(bmode_pw, [x_grid[0], x_grid[-1], z_grid[-1], z_grid[0]],
            title='Coherent Compound (3 angles)', ax=axes[-1])
plt.tight_layout()
plt.show()

## 小结

- 点散射体仿真 + DAS 波束合成能正确成像
- B-mode 图像显示: 单个点、点阵、囊肿三种 phantom 均可重建
- 横向分辨率受 F-number 和波长限制
- 平面波 + 角度复合可提高图像质量
- 下一步: 训练 UNet 进行超分辨/去噪